<a id="top"></a>
# Paper 3 — Evaluasi Terpisah & Komprehensif: `Ortho-Unified` vs. `B1_M4_CanonBridge`, `M4_GeoFiLM`, `E_CLBRIDGE`, `A0`, & `Gautam_Qwen2.5VL_7B`

Notebook evaluasi terpisah ini membaca `eval/predictions.json`, `eval/canon_preds.json`, dan `eval/ortho_diag.json` yang dihasilkan oleh **`06_Ortho_Unified_Training.ipynb`**, lalu menjalankan **dua rangkaian evaluasi lengkap** secara *apple-to-apple* di atas **15.955 baris test set** (`31.622` aspek klinis):

1. **BAGIAN 1 (`[E1]`–`[E5]`): Evaluasi Cepat CPU & Diagnostik Arsitektural (~30 detik, Tanpa GPU Judge)**
   - **Token-F1 Overall**, per tingkat kompleksitas (`c1`, `c2`, `c3`), dan per **18 kategori klinis** (menggunakan fungsi `token_f1` yang persis sama dengan notebook 01 & 05)
   - **Metrik Spasial 9-Grid (`abnormality_location`, `instrument_location`, `landmark_location`):** Multi-label 9-Zone Jaccard, Zone Recall, Zone Precision, & **$\ge 3$-Zone Hedge Collapse Rate**
   - **Metrik Ordinal `polyp_size`:** Exact Tier Accuracy, Within-1 Tier Accuracy, Normalized MAE, & Quadratic Weighted Kappa (**QWK**)
   - **Uji Signifikansi McNemar Berpasangan** (`Ortho-Unified` vs `Gautam_Qwen2.5VL_7B`, `B1_M4_CanonBridge`, `M4_GeoFiLM`, `E_CLBRIDGE`, `A0`)
   - **Diagnostik Kepala Kanonik, Query-Aspect Router $r_c(Q)$, Kepala CORN, & Ortogonalitas Gram-Schmidt** (`Ortho-Unified` dibandingkan langsung dengan `B1_M4_CanonBridge`).
2. **BAGIAN 2 (`[J1]`–`[J5]`): Hybrid L1 Deterministic + Dual-LLM Judge (`Qwen2.5-7B-Instruct` & `Llama-3.1-8B-Instruct`)**
   - Menggunakan **mesin deduplikasi, prompt few-shot `JUDGE_TMPL`, dan Global Pool (`global_pool_qwen25_7b.json` & `global_pool_llama31_8b.json`, 67.360 entri)** yang 100% identik dengan `02_GeoFiLM_Dual_LLM_Judge_Evaluation.ipynb`.
   - Seluruh 9 model pembanding (`A0`, `E_CLBRIDGE`, `Gautam_Qwen2.5VL_7B`, `M1`–`M4`, `GeoFiLMpp_U4`, `Bridge_B1_M4_CanonBridge`) **otomatis disalin dari cache tanpa makan waktu GPU (0 menit)**, dan hanya baris L2/L3 unik baru milik `Ortho-Unified` yang dinilai di GPU.

# BAGIAN 1 — SETUP & EVALUASI CEPAT CPU + DIAGNOSTIK KEPALA
## [E1] Install paket & mount Google Drive

In [1]:
# [E1] Install paket & mount Google Drive
!pip install -q -U "transformers>=4.46.0,<4.52.0" "accelerate>=0.34.0" bitsandbytes safetensors datasets scipy scikit-learn seaborn matplotlib pandas tqdm

import os, sys, json, re, ast, math, time, gc, shutil, warnings
from pathlib import Path
from collections import Counter, defaultdict
import numpy as np
import pandas as pd
from scipy.stats import binomtest
from sklearn.metrics import cohen_kappa_score
from tqdm.auto import tqdm
warnings.filterwarnings("ignore")

try:
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)
    IS_COLAB = True
except ImportError:
    IS_COLAB = False

BASE       = Path("/content/drive/MyDrive/DOKTORAL/KvasirVQAX1") if IS_COLAB else Path(r"g:/My Drive/DOKTORAL/KvasirVQAX1")
POLYPSIZE  = BASE / "PolypSize"
PAPER2_DIR = BASE / "GeoFiLM_Paper2"
CORE_DIR   = PAPER2_DIR
P2_EXP_DIR = BASE / "experiments/Paper2_GeoFiLM_r32"
P2PLUS_DIR = BASE / "experiments/Paper2plus_GeoFiLMpp_r32"
BRIDGE_DIR = BASE / "experiments/Paper2plus_Bridge_r32"
ORTHO_DIR  = BASE / "experiments/Paper3_Ortho_Unified_r32"
S2_R32     = BASE / "experiments/Stage2_CORAL_linear/prediction_rank32"
S4_DIR     = BASE / "experiments/Stage4_SACORN_and_Bridge_r32"
CMP_DIR    = BASE / "experiments/Comparison_Gautam"

# Folder cache global dari Paper 1 & Paper 2 (untuk reuse 67.360 entri) & folder output khusus Paper 3
SHARED_JDIR = POLYPSIZE / "dual_judge_eval"
P2_JDIR     = PAPER2_DIR / "dual_judge_eval"
DUAL_JDIR   = PAPER2_DIR / "ortho_unified_eval"
DUAL_JDIR.mkdir(parents=True, exist_ok=True)

HF_CACHE = CMP_DIR / "hf_cache"
HF_CACHE.mkdir(parents=True, exist_ok=True)
os.environ["HF_HOME"] = str(HF_CACHE)

if str(CORE_DIR) not in sys.path:
    sys.path.insert(0, str(CORE_DIR))
import importlib, ortho_unified_core as _ouc
importlib.reload(_ouc)
from ortho_unified_core import (
    as_list, atoms, as_class_list, aspect_pairs, CanonLabeler,
    POLYP_SIZE_ORDERED_TIERS, align_prose_with_heads, load_aligner_kb
)
print("BASE      :", BASE)
print("DUAL_JDIR :", DUAL_JDIR)

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires pandas==2.2.3, but you have pandas 3.1.1 which is incompatible.
dask-cudf-cu13 26.6.0 requires pandas<2.4.0,>=2.0, but you have pandas 3.1.1 which is incompatible.
gradio-client 2.7.2 requires huggingface-hub<3.0,>=1.16.0, but you have huggingface-hub 0.36.2 which is incompatible.
diffusers 0.42.1 requires huggingface-hub<3.0,>=1.32.0, but you have huggingface-hub 0.36.2 which is incompatible.
cudf-cu13 26.6.0 requires pandas<2.4.0,>=2.0, but you have pandas 3.1.1 which is incompatible.
gradio 6.31.0 requires huggingface-hub<3.0,>=1.16.0, but you have huggingface-hub 0.36.2 which is incompatible.


/usr/local/lib/python3.13/dist-packages/pandas/core/arrays/masked.py:63: UserWarning: Pandas requires version '1.5.0' or newer of 'bottleneck' (version '1.4.2' currently installed).
  from pandas.core import (


Mounted at /content/drive
BASE      : /content/drive/MyDrive/DOKTORAL/KvasirVQAX1
DUAL_JDIR : /content/drive/MyDrive/DOKTORAL/KvasirVQAX1/GeoFiLM_Paper2/ortho_unified_eval


## [E2] Muat Test Set (`15.955` baris), Auto-Discover Semua Model, & Sinkronisasi Cache Judge

In [2]:
# [E2] Muat Test Set (15.955 baris), Auto-Discover Semua Prediksi Model, & Sinkronisasi Cache Global
from datasets import load_from_disk

ds = load_from_disk(str(BASE / "test"))
te = ds.to_pandas()
ANS_COL, QCLASS_COL, IMGID_COL = "answer", "question_class", "img_id"
keep = [ANS_COL, QCLASS_COL, IMGID_COL, "original", "question", "complexity"]
rows = ds.remove_columns([c for c in ds.column_names if c not in keep]).to_list()
print(f"✓ Total test rows loaded: {len(te)}")
assert len(te) == 15955

MODEL_PATHS = {
    "A0_VQAonly_r32":     S2_R32 / "20260916-060029_VQAonly-r32_a0.0_s42/eval/predictions.json",
    "E_CLBRIDGE_r32":     S4_DIR / "20261023-060251_E-clbridge-r32_a0.05_s42/eval/predictions.json",
    "Gautam_Qwen2.5VL_7B": CMP_DIR / "predictions_qwenft.json",
}

for v_tag in ["M1_GeoPE", "M2_GeoDiff", "M3_StaticGate", "M4_GeoFiLM"]:
    if P2_EXP_DIR.exists():
        cands = [p / "eval/predictions.json" for p in sorted(P2_EXP_DIR.glob(f"*{v_tag}*")) if (p / "eval/predictions.json").exists()]
        if cands:
            MODEL_PATHS[f"{v_tag}_r32"] = cands[-1]

if BRIDGE_DIR.exists():
    for v_tag in ["B0_M4_repro", "B1_M4_CanonBridge"]:
        runs = [p for p in sorted(BRIDGE_DIR.glob(f"*{v_tag}*")) if (p / "eval/predictions.json").exists()]
        if runs:
            MODEL_PATHS[f"Bridge_{v_tag}"] = runs[-1] / "eval/predictions.json"

MODEL_RUN_DIRS = {}
if BRIDGE_DIR.exists():
    b1_runs = [p for p in sorted(BRIDGE_DIR.glob("*B1_M4_CanonBridge*")) if (p / "eval/predictions.json").exists()]
    if b1_runs:
        MODEL_RUN_DIRS["Bridge_B1_M4_CanonBridge"] = b1_runs[-1]

# Deteksi otomatis seluruh run Ortho-Unified (Paper 3) dari experiments/Paper3_Ortho_Unified_r32
if ORTHO_DIR.exists():
    for v_tag in ["O0_M4_repro", "O1_Ortho_NoPatchFiLM", "O2_Ortho_Unified_Full"]:
        runs = [p for p in sorted(ORTHO_DIR.glob(f"*{v_tag}*")) if (p / "eval/predictions.json").exists()]
        if runs:
            rdir = runs[-1]
            # Pastikan Closed-Loop Self-Consistency Readout Alignment telah diterapkan pada predictions.json
            raw_p_file = rdir / "eval/predictions_raw.json"
            pred_p_file = rdir / "eval/predictions.json"
            canon_p_file = rdir / "eval/canon_preds.json"
            if canon_p_file.exists():
                if not raw_p_file.exists():
                    shutil.copy2(pred_p_file, raw_p_file)
                _raw_d = json.loads(raw_p_file.read_text(encoding="utf-8"))
                _can_d = json.loads(canon_p_file.read_text(encoding="utf-8"))
                _kb = load_aligner_kb(PAPER2_DIR / "ortho_aligner_kb.json")
                _aln_d = {
                    str(i): align_prose_with_heads(str(te["question"].iloc[i]), _raw_d[str(i)], _can_d.get(str(i), {}), kb=_kb)
                    for i in range(len(te))
                }
                pred_p_file.write_text(json.dumps(_aln_d, indent=1, ensure_ascii=False), encoding="utf-8")
                MODEL_PATHS["Ortho_O2_Raw_Decoder"] = raw_p_file
            m_key = f"Ortho_{v_tag}"
            MODEL_PATHS[m_key] = pred_p_file
            MODEL_RUN_DIRS[m_key] = rdir

PREDS = {}
print("=== VERIFIKASI PREDIKSI 15.955 BARIS ===")
for name, p in MODEL_PATHS.items():
    if not p.exists():
        print(f"  [SKIP] {name:26s} : File belum ada ({p})")
        continue
    raw = json.loads(p.read_text(encoding="utf-8"))
    if isinstance(raw, dict):
        d = {int(k): str(v).strip() for k, v in raw.items()}
    else:
        d = {int(rec.get("idx", i)): str(rec.get("pred", rec.get("prediction", rec.get("pred_full", "")))).strip()
             for i, rec in enumerate(raw)}
    PREDS[name] = d
    non_empty = sum(1 for v in d.values() if v)
    print(f"  [OK]   {name:26s} : {len(d):5d} baris ({non_empty:5d} terisi) <- {p.relative_to(BASE)}")

# Salin cache Global Pool & cache per-model dari P2_JDIR / SHARED_JDIR ke DUAL_JDIR (0 menit GPU untuk baseline!)
for j_tag in ["qwen25_7b", "llama31_8b"]:
    dst_pool = DUAL_JDIR / f"global_pool_{j_tag}.json"
    merged_pool = {}
    for src_dir in [SHARED_JDIR, P2_JDIR, DUAL_JDIR]:
        sp = src_dir / f"global_pool_{j_tag}.json"
        if sp.exists():
            try:
                merged_pool.update(json.loads(sp.read_text(encoding="utf-8")))
            except Exception:
                pass
    if merged_pool:
        dst_pool.write_text(json.dumps(merged_pool), encoding="utf-8")
        print(f"  [SYNC] Global Pool {dst_pool.name} siap ({len(merged_pool):,} entri unik)")
    for mname in PREDS:
        dst_mc = DUAL_JDIR / f"judge_{j_tag}_{mname}.json"
        if not dst_mc.exists():
            for src_dir in [P2_JDIR, SHARED_JDIR]:
                src_mc = src_dir / f"judge_{j_tag}_{mname}.json"
                if src_mc.exists():
                    shutil.copy2(src_mc, dst_mc)
                    print(f"  [SYNC] Menyalin cache baseline {src_mc.name} (100% siap tanpa GPU)")
                    break

ORTHO_KEYS = [k for k in PREDS if k.startswith("Ortho_")]
PRIMARY_MODEL = ORTHO_KEYS[-1] if ORTHO_KEYS else ("Bridge_B1_M4_CanonBridge" if "Bridge_B1_M4_CanonBridge" in PREDS else list(PREDS.keys())[-1])
print(f"\n>>> PRIMARY_MODEL untuk evaluasi: {PRIMARY_MODEL} <<<")

✓ Total test rows loaded: 15955
=== VERIFIKASI PREDIKSI 15.955 BARIS ===
  [OK]   A0_VQAonly_r32             : 15955 baris (15955 terisi) <- experiments/Stage2_CORAL_linear/prediction_rank32/20260916-060029_VQAonly-r32_a0.0_s42/eval/predictions.json
  [OK]   E_CLBRIDGE_r32             : 15955 baris (15955 terisi) <- experiments/Stage4_SACORN_and_Bridge_r32/20261023-060251_E-clbridge-r32_a0.05_s42/eval/predictions.json
  [OK]   Gautam_Qwen2.5VL_7B        : 15955 baris (15955 terisi) <- experiments/Comparison_Gautam/predictions_qwenft.json
  [OK]   M1_GeoPE_r32               : 15955 baris (15955 terisi) <- experiments/Paper2_GeoFiLM_r32/20261030-212438_M1_GeoPE_r32_s42/eval/predictions.json
  [OK]   M2_GeoDiff_r32             : 15955 baris (15955 terisi) <- experiments/Paper2_GeoFiLM_r32/20261031-142834_M2_GeoDiff_r32_s42/eval/predictions.json
  [OK]   M3_StaticGate_r32          : 15955 baris (15955 terisi) <- experiments/Paper2_GeoFiLM_r32/20261031-235417_M3_StaticGate_r32_s42/eval/pred

## [E3] Token-F1 Overall, Per Kompleksitas, Per 18 Kategori, & Uji McNemar

In [3]:
# [E3] Token-F1 (fungsi PERSIS notebook 01 & 05, \w+) + Per Kompleksitas + 18 Kategori + McNemar
def token_f1(pred, gold):
    p_toks = re.findall(r"\w+", str(pred).lower())
    g_toks = re.findall(r"\w+", str(gold).lower())
    if not p_toks or not g_toks:
        return float(p_toks == g_toks)
    common = sum((Counter(p_toks) & Counter(g_toks)).values())
    if common == 0:
        return 0.0
    prec, rec = common / len(p_toks), common / len(g_toks)
    return 2 * prec * rec / (prec + rec)

gold = te["answer"].astype(str).tolist()
cats = te["question_class"].map(as_class_list)
f1 = {k: np.array([token_f1(d.get(i, ""), gold[i]) for i in range(len(te))]) for k, d in PREDS.items()}

print("=" * 95)
print(f" TOKEN-F1 OVERALL (15.955 Test Rows) — Primary: {PRIMARY_MODEL}")
print("=" * 95)
for k, v in f1.items():
    extra = f"   (Δ {PRIMARY_MODEL} vs {k}: {100*(f1[PRIMARY_MODEL].mean()-v.mean()):+.2f}pp)" if k != PRIMARY_MODEL else "   <-- [PRIMARY]"
    print(f"  {k:28s} {100*v.mean():6.2f}%{extra}")

if "complexity" in te.columns:
    print("\nPer kompleksitas (c1 / c2 / c3):")
    for c in sorted(te["complexity"].unique()):
        msk = (te["complexity"] == c).to_numpy()
        parts = [f"{k}: {100*f1[k][msk].mean():5.2f}%" for k in f1]
        print(f"  c{c} (n={msk.sum():5d}) | " + " | ".join(parts))

print("\nPer kategori (18 kategori klinis):")
cat_rows = []
for c in sorted({x for L in cats for x in L}):
    msk = cats.map(lambda L: c in L).to_numpy()
    r = {"kategori": c, "n": int(msk.sum())}
    for k in f1:
        r[k] = round(100 * f1[k][msk].mean(), 2)
    if "Bridge_B1_M4_CanonBridge" in f1 and PRIMARY_MODEL in f1:
        r["Δ_vs_B1"] = round(r[PRIMARY_MODEL] - r["Bridge_B1_M4_CanonBridge"], 2)
    if "M4_GeoFiLM_r32" in f1 and PRIMARY_MODEL in f1:
        r["Δ_vs_M4"] = round(r[PRIMARY_MODEL] - r["M4_GeoFiLM_r32"], 2)
    cat_rows.append(r)
df_cat_f1 = pd.DataFrame(cat_rows)
display(df_cat_f1)
df_cat_f1.to_csv(DUAL_JDIR / "token_f1_per_category.csv", index=False)

print(f"\nUji McNemar Berpasangan untuk {PRIMARY_MODEL} (token-F1 >= 0,5 dianggap benar):")
mc_rows = []
a_bin = (f1[PRIMARY_MODEL] >= 0.5).astype(int)
for k in f1:
    if k == PRIMARY_MODEL:
        continue
    b_bin = (f1[k] >= 0.5).astype(int)
    n01 = int(((a_bin == 1) & (b_bin == 0)).sum())
    n10 = int(((a_bin == 0) & (b_bin == 1)).sum())
    pval = binomtest(n01, n01 + n10, 0.5).pvalue if (n01 + n10) else 1.0
    mc_rows.append({"vs_model": k, "fix (n01)": n01, "break (n10)": n10,
                    "net_rows": n01 - n10, "delta_pp": round(100 * (a_bin.mean() - b_bin.mean()), 2),
                    "p_value": pval})
    print(f"  vs {k:28s} fix {n01:5d} | break {n10:5d} | net {n01-n10:+5d} | Δ {100*(a_bin.mean()-b_bin.mean()):+.2f}pp | p={pval:.3e}")
pd.DataFrame(mc_rows).to_csv(DUAL_JDIR / "mcnemar_tokenf1.csv", index=False)

 TOKEN-F1 OVERALL (15.955 Test Rows) — Primary: Ortho_O2_Ortho_Unified_Full
  A0_VQAonly_r32                65.64%   (Δ Ortho_O2_Ortho_Unified_Full vs A0_VQAonly_r32: +3.52pp)
  E_CLBRIDGE_r32                65.30%   (Δ Ortho_O2_Ortho_Unified_Full vs E_CLBRIDGE_r32: +3.86pp)
  Gautam_Qwen2.5VL_7B           71.09%   (Δ Ortho_O2_Ortho_Unified_Full vs Gautam_Qwen2.5VL_7B: -1.92pp)
  M1_GeoPE_r32                  66.91%   (Δ Ortho_O2_Ortho_Unified_Full vs M1_GeoPE_r32: +2.26pp)
  M2_GeoDiff_r32                66.87%   (Δ Ortho_O2_Ortho_Unified_Full vs M2_GeoDiff_r32: +2.29pp)
  M3_StaticGate_r32             67.15%   (Δ Ortho_O2_Ortho_Unified_Full vs M3_StaticGate_r32: +2.02pp)
  M4_GeoFiLM_r32                68.29%   (Δ Ortho_O2_Ortho_Unified_Full vs M4_GeoFiLM_r32: +0.87pp)
  Bridge_B1_M4_CanonBridge      68.90%   (Δ Ortho_O2_Ortho_Unified_Full vs Bridge_B1_M4_CanonBridge: +0.27pp)
  Ortho_O2_Ortho_Unified_Full   69.17%   <-- [PRIMARY]

Per kompleksitas (c1 / c2 / c3):
  c1 (n= 5496) | A0

,kategori,n,A0_VQAonly_r32,E_CLBRIDGE_r32,Gautam_Qwen2.5VL_7B,M1_GeoPE_r32,M2_GeoDiff_r32,M3_StaticGate_r32,M4_GeoFiLM_r32,Bridge_B1_M4_CanonBridge,Ortho_O2_Ortho_Unified_Full,Δ_vs_B1,Δ_vs_M4
0,abnormality_color,1862,58.98,59.60,66.09,61.33,60.65,61.02,61.52,62.73,62.39,-0.34,0.87
1,abnormality_location,1901,59.69,61.67,68.68,61.73,61.65,61.29,62.15,63.09,63.68,0.59,1.53
2,abnormality_presence,1823,60.94,61.92,68.71,62.81,63.70,64.23,64.75,65.82,65.35,-0.47,0.60
3,box_artifact_presence,2319,73.44,74.02,77.08,74.60,74.53,74.67,74.70,75.38,75.47,0.09,0.77
4,finding_count,2115,62.08,61.74,68.66,62.87,62.84,63.48,65.14,66.38,65.31,-1.07,0.17
5,finding_presence,237,78.78,70.24,58.25,75.92,76.45,77.08,77.20,76.54,77.78,1.24,0.58
6,instrument_count,2208,71.42,70.38,76.11,73.17,72.64,73.11,73.61,74.30,74.37,0.07,0.76
7,instrument_location,1913,68.06,68.26,71.97,68.85,68.76,68.91,69.23,69.92,69.59,-0.33,0.36
8,instrument_presence,1837,63.23,63.10,68.84,63.73,64.54,64.60,66.48,66.98,67.23,0.25,0.75
9,landmark_color,55,57.80,55.11,58.05,55.76,54.75,56.51,58.66,58.34,57.62,-0.72,-1.04



Uji McNemar Berpasangan untuk Ortho_O2_Ortho_Unified_Full (token-F1 >= 0,5 dianggap benar):
  vs A0_VQAonly_r32               fix  1321 | break   673 | net  +648 | Δ +4.06pp | p=2.073e-48
  vs E_CLBRIDGE_r32               fix  1421 | break   732 | net  +689 | Δ +4.32pp | p=1.383e-50
  vs Gautam_Qwen2.5VL_7B          fix  1130 | break  1449 | net  -319 | Δ -2.00pp | p=3.620e-10
  vs M1_GeoPE_r32                 fix  1069 | break   665 | net  +404 | Δ +2.53pp | p=2.433e-22
  vs M2_GeoDiff_r32               fix  1099 | break   666 | net  +433 | Δ +2.71pp | p=4.905e-25
  vs M3_StaticGate_r32            fix  1062 | break   675 | net  +387 | Δ +2.43pp | p=1.405e-20
  vs M4_GeoFiLM_r32               fix   785 | break   644 | net  +141 | Δ +0.88pp | p=2.103e-04
  vs Bridge_B1_M4_CanonBridge     fix   697 | break   690 | net    +7 | Δ +0.04pp | p=8.720e-01


## [E4] Evaluasi Khusus 9-Grid Spatial Location & Ordinal `polyp_size` (Full & $N=742$)

In [4]:
# [E4] Evaluasi 9-Grid Spatial (Jaccard, Recall, Hedge Rate >=3) & Ordinal polyp_size (Exact, Within-1, MAE, QWK)
ZONES_9 = [
    "upper-left", "upper-center", "upper-right",
    "center-left", "center", "center-right",
    "lower-left", "lower-center", "lower-right",
]
ZONE_PATS = {
    z: re.compile(r"\b" + z.replace("-", r"[\s\-]+") + r"\b", re.I)
    for z in sorted(ZONES_9, key=len, reverse=True)
}

def extract_zones(text: str) -> set:
    s = str(text).lower()
    out = set()
    for z in sorted(ZONES_9, key=len, reverse=True):
        if ZONE_PATS[z].search(s):
            out.add(z)
            s = ZONE_PATS[z].sub(" ", s)
    return out

SIZE_PATS = [
    (0, re.compile(r"(<\s*5\s*mm|less than 5\s*mm|under 5\s*mm|diminutive)", re.I)),
    (1, re.compile(r"(5\s*[\-–to]+\s*10\s*mm|between 5 and 10\s*mm)", re.I)),
    (2, re.compile(r"(11\s*[\-–to]+\s*20\s*mm|10\s*[\-–to]+\s*20\s*mm|between 1[01] and 20\s*mm)", re.I)),
    (3, re.compile(r"(>\s*20\s*mm|more than 20\s*mm|greater than 20\s*mm|over 20\s*mm)", re.I)),
]

def extract_size_ranks(text: str) -> list:
    return sorted({rk for rk, pat in SIZE_PATS if pat.search(str(text))})

LOC_CATS = {"abnormality_location", "instrument_location", "landmark_location"}
tier_map = {t: idx for idx, t in enumerate(POLYP_SIZE_ORDERED_TIERS)}

spat_summary, size_summary = [], []

for m_name, preds_d in PREDS.items():
    jacc_l, rec_l, prec_l, hedge_l, nzone_l = [], [], [], [], []
    sz_gt_all, sz_pr_all, sz_gt_s1, sz_pr_s1 = [], [], [], []

    for i in range(len(te)):
        row = te.iloc[i]
        pred_txt = str(preds_d.get(i, ""))
        gt_txt   = str(row["answer"])
        row_cats = as_class_list(row["question_class"])

        if any(c in LOC_CATS for c in row_cats):
            gz = extract_zones(gt_txt)
            pz = extract_zones(pred_txt)
            if gz:
                jacc_l.append(len(gz & pz) / max(len(gz | pz), 1))
                rec_l.append(len(gz & pz) / len(gz))
                prec_l.append(len(gz & pz) / len(pz) if pz else 0.0)
                hedge_l.append(1.0 if len(pz) >= 3 else 0.0)
                nzone_l.append(len(pz))

        for c, a in (aspect_pairs(row["question_class"], row["original"]) or []):
            if c == "polyp_size":
                gt_rks = [tier_map[at] for at in atoms(a) if at in tier_map]
                if gt_rks:
                    g_rk = max(gt_rks)
                    p_rks = extract_size_ranks(pred_txt)
                    p_rk = max(p_rks) if p_rks else 1
                    hit_exact = int(bool(p_rks) and (g_rk in p_rks))
                    sz_gt_all.append(g_rk)
                    sz_pr_all.append(g_rk if hit_exact else p_rk)
                    if len(gt_rks) == 1 and len(row_cats) == 1:
                        sz_gt_s1.append(g_rk)
                        sz_pr_s1.append(g_rk if hit_exact else p_rk)

    spat_summary.append({
        "Model": m_name,
        "9-Zone Jaccard (%)": round(100 * np.mean(jacc_l), 2) if jacc_l else 0.0,
        "Zone Recall (%)": round(100 * np.mean(rec_l), 2) if rec_l else 0.0,
        "Zone Precision (%)": round(100 * np.mean(prec_l), 2) if prec_l else 0.0,
        "Hedge >=3 Zones (%)": round(100 * np.mean(hedge_l), 2) if hedge_l else 0.0,
        "Avg Zones Pred": round(float(np.mean(nzone_l)), 2) if nzone_l else 0.0,
    })

    gt_a, pr_a = np.array(sz_gt_all), np.array(sz_pr_all)
    gt_s, pr_s = np.array(sz_gt_s1), np.array(sz_pr_s1)
    size_summary.append({
        "Model": m_name,
        "All_N": len(gt_a),
        "All_Exact (%)": round(100 * float(np.mean(gt_a == pr_a)), 2) if len(gt_a) else 0.0,
        "All_Within1 (%)": round(100 * float(np.mean(np.abs(gt_a - pr_a) <= 1)), 2) if len(gt_a) else 0.0,
        "All_Norm_MAE": round(float(np.mean(np.abs(gt_a - pr_a)) / 3.0), 4) if len(gt_a) else 0.0,
        "All_QWK": round(float(cohen_kappa_score(gt_a, pr_a, weights="quadratic")), 4) if len(gt_a) > 1 else 0.0,
        "Single_N": len(gt_s),
        "Single_Exact (%)": round(100 * float(np.mean(gt_s == pr_s)), 2) if len(gt_s) else 0.0,
    })

df_spat = pd.DataFrame(spat_summary)
df_size = pd.DataFrame(size_summary)
print("=== 9-GRID SPATIAL LOCATION EVALUATION ===")
display(df_spat)
print("\n=== ORDINAL POLYP SIZE EVALUATION ===")
display(df_size)
df_spat.to_csv(DUAL_JDIR / "spatial_9grid_metrics.csv", index=False)
df_size.to_csv(DUAL_JDIR / "polyp_size_ordinal_metrics.csv", index=False)

=== 9-GRID SPATIAL LOCATION EVALUATION ===


,Model,9-Zone Jaccard (%),Zone Recall (%),Zone Precision (%),Hedge >=3 Zones (%),Avg Zones Pred
0,A0_VQAonly_r32,8.03,9.05,9.64,1.42,0.20
1,E_CLBRIDGE_r32,9.03,10.92,12.07,4.17,0.33
2,Gautam_Qwen2.5VL_7B,34.46,42.71,41.11,18.72,1.55
3,M1_GeoPE_r32,9.65,11.57,13.45,3.97,0.38
4,M2_GeoDiff_r32,9.47,11.53,12.58,4.88,0.39
5,M3_StaticGate_r32,7.42,8.86,10.76,1.63,0.24
6,M4_GeoFiLM_r32,12.96,16.34,17.29,5.90,0.54
7,Bridge_B1_M4_CanonBridge,15.93,19.51,21.11,7.63,0.64
8,Ortho_O2_Ortho_Unified_Full,18.76,21.84,25.27,6.61,0.65



=== ORDINAL POLYP SIZE EVALUATION ===


,Model,All_N,All_Exact (%),All_Within1 (%),All_Norm_MAE,All_QWK,Single_N,Single_Exact (%)
0,A0_VQAonly_r32,733,31.38,82.95,0.2856,0.0000,106,23.58
1,E_CLBRIDGE_r32,733,31.38,82.95,0.2856,0.0000,106,23.58
2,Gautam_Qwen2.5VL_7B,733,34.11,83.08,0.2760,0.0237,106,23.58
3,M1_GeoPE_r32,733,31.38,82.95,0.2856,0.0000,106,23.58
4,M2_GeoDiff_r32,733,31.38,82.95,0.2856,0.0000,106,23.58
5,M3_StaticGate_r32,733,31.38,82.95,0.2856,0.0000,106,23.58
6,M4_GeoFiLM_r32,733,31.38,82.95,0.2856,0.0000,106,23.58
7,Bridge_B1_M4_CanonBridge,733,31.38,82.95,0.2856,0.0000,106,23.58
8,Ortho_O2_Ortho_Unified_Full,733,31.38,82.95,0.2856,0.0000,106,23.58


## [E5] Diagnostik Kepala Kanonik, Query-Aspect Router, CORN, & Gram-Schmidt (`Ortho-Unified` vs `B1_M4_CanonBridge`)

In [5]:
# [E5] Diagnostik 16 Kepala Kanonik di TEST Set: Ortho-Unified vs B1_M4_CanonBridge
ortho_run = MODEL_RUN_DIRS.get(PRIMARY_MODEL)
b1_run    = MODEL_RUN_DIRS.get("Bridge_B1_M4_CanonBridge")

if ortho_run is not None and (ortho_run / "eval/canon_preds.json").exists():
    cfg_o = json.loads((ortho_run / "config.json").read_text())
    lab_o = CanonLabeler(cfg_o["canon_specs"])
    cp_ortho = {int(k): v for k, v in json.loads((ortho_run / "eval/canon_preds.json").read_text()).items()}
    cp_b1 = (
        {int(k): v for k, v in json.loads((b1_run / "eval/canon_preds.json").read_text()).items()}
        if (b1_run is not None and (b1_run / "eval/canon_preds.json").exists()) else None
    )

    cnt_o = {c: [0, 0] for c in lab_o.cats}
    cnt_b = {c: [0, 0] for c in lab_o.cats}
    for i, (q, o) in enumerate(zip(te["question_class"], te["original"])):
        for ci, idx in lab_o.encode(q, o):
            c = lab_o.cats[ci]
            s0 = lab_o.slices[ci][0]
            gold_atoms = {cfg_o["canon_specs"][c]["atoms"][j - s0] for j in idx}
            pred_o = set(cp_ortho.get(i, {}).get(c, []))
            cnt_o[c][0] += 1
            cnt_o[c][1] += int(gold_atoms == pred_o)
            if cp_b1 is not None:
                pred_b = set(cp_b1.get(i, {}).get(c, []))
                cnt_b[c][0] += 1
                cnt_b[c][1] += int(gold_atoms == pred_b)

    head_rows = []
    for c in lab_o.cats:
        n_c, ok_o = cnt_o[c]
        acc_o = round(100 * ok_o / max(n_c, 1), 2)
        acc_b = round(100 * cnt_b[c][1] / max(cnt_b[c][0], 1), 2) if cp_b1 is not None else None
        head_rows.append({
            "kategori": c,
            "n_test": n_c,
            "B1_CanonBridge_Head_%": acc_b,
            "Ortho_Unified_Head_%": acc_o,
            "Δ_Head_pp": round(acc_o - acc_b, 2) if acc_b is not None else None,
            "multi": cfg_o["canon_specs"][c]["multi"],
        })
    df_head_cmp = pd.DataFrame(head_rows).sort_values("Ortho_Unified_Head_%")
    print("=== PERBANDINGAN AKURASI 16 KEPALA ENCODER DI TEST SET (Ortho-Unified vs B1_M4_CanonBridge) ===")
    display(df_head_cmp)
    df_head_cmp.to_csv(DUAL_JDIR / "encoder_head_accuracy_ortho_vs_b1.csv", index=False)

    diag_p = ortho_run / "eval/ortho_diag.json"
    if diag_p.exists():
        dg_all = {int(k): v for k, v in json.loads(diag_p.read_text()).items()}
        r_tp, r_fp, r_fn = 0, 0, 0
        cos_g_vals, cos_s_vals = [], []
        for i, (q, o) in enumerate(zip(te["question_class"], te["original"])):
            d_i = dg_all.get(i, {})
            if "cos_glob" in d_i:
                cos_g_vals.append(d_i["cos_glob"])
                cos_s_vals.append(d_i["cos_spat"])
            asked_cats = {lab_o.cats[ci] for ci, _ in lab_o.encode(q, o)}
            pred_cats  = set(d_i.get("router_cats", []))
            for c in lab_o.cats:
                a_in = c in asked_cats
                p_in = c in pred_cats
                if a_in and p_in: r_tp += 1
                elif (not a_in) and p_in: r_fp += 1
                elif a_in and (not p_in): r_fn += 1
        r_prec = 100 * r_tp / max(r_tp + r_fp, 1)
        r_rec  = 100 * r_tp / max(r_tp + r_fn, 1)
        r_f1   = 2 * r_prec * r_rec / max(r_prec + r_rec, 1e-6)
        print(f"\n[Query-Aspect Router r_c(Q)] Precision: {r_prec:.2f}% | Recall: {r_rec:.2f}% | F1: {r_f1:.2f}%")
        if cos_g_vals:
            print(f"[Gram-Schmidt Orthogonality] Mean |cos(glob_perp, e_total)| = {np.mean(cos_g_vals):.2e} | Mean |cos(inj_perp, pe_2d)| = {np.mean(cos_s_vals):.2e}")
else:
    print("File canon_preds.json belum tersedia; jalankan [E2] pada 06_Ortho_Unified_Training.ipynb terlebih dahulu.")

=== PERBANDINGAN AKURASI 16 KEPALA ENCODER DI TEST SET (Ortho-Unified vs B1_M4_CanonBridge) ===


,kategori,n_test,B1_CanonBridge_Head_%,Ortho_Unified_Head_%,Δ_Head_pp,multi
0,abnormality_color,1862,49.46,43.39,-6.07,True
8,landmark_location,1224,55.15,54.90,-0.25,True
1,abnormality_location,1897,62.84,63.78,0.94,True
13,polyp_type,1944,73.92,72.74,-1.18,True
6,instrument_location,1913,69.26,79.98,10.72,True
12,polyp_size,1925,72.73,80.26,7.53,True
4,finding_count,2108,93.12,92.08,-1.04,False
9,landmark_presence,1260,92.70,92.78,0.08,False
7,instrument_presence,1837,90.75,95.59,4.84,True
3,box_artifact_presence,2319,97.02,97.24,0.22,False



[Query-Aspect Router r_c(Q)] Precision: 21.80% | Recall: 10.18% | F1: 13.88%
[Gram-Schmidt Orthogonality] Mean |cos(glob_perp, e_total)| = 1.50e-08 | Mean |cos(inj_perp, pe_2d)| = 4.87e-08


---
# BAGIAN 2 — PROTOKOL PENUH DUAL-LLM JUDGE (`Qwen2.5-7B-Instruct` & `Llama-3.1-8B-Instruct`)
<a id="part-judge"></a>

100% identik dengan protokol `02_GeoFiLM_Dual_LLM_Judge_Evaluation.ipynb`:
- **`[J1]`**: Scorer Deterministik Level-1 (`5.496` pertanyaan tunggal dihitung instan di CPU) + Mesin Deduplikasi Global (`JUDGE_TMPL` few-shot) yang otomatis mewarisi `67.360` entri dari `GeoFiLM_Paper2/dual_judge_eval/global_pool_*.json`.
- **`[J2]`**: Eksekusi Judge 1 (`Qwen/Qwen2.5-7B-Instruct`) hanya pada baris unik baru yang belum ada di pool.
- **`[J3]`**: Eksekusi Judge 2 (`NousResearch/Meta-Llama-3.1-8B-Instruct`) hanya pada baris unik baru yang belum ada di pool.
- **`[J4]`**: Kompilasi tabel utama 18 Kategori $\times$ 3 Level Kompleksitas (`31.622` aspek), Inter-Judge Cohen's $\kappa$, dan Uji Signifikansi McNemar Berpasangan (`Ortho-Unified` vs `Gautam_Qwen2.5VL_7B`, `B1_M4_CanonBridge`, `M4_GeoFiLM`, `E_CLBRIDGE`, `A0`).

## [J1] Scorer Deterministik Level-1 (`5.496` Baris) & Mesin Global Deduplication Judge

In [6]:
# [J1] Scorer Deterministik Level-1 (5.496 baris) & Mesin Deduplikasi Global (identik notebook 02 [J3]-[J4])
def qclass(r):
    qc = r[QCLASS_COL]
    if isinstance(qc, (list, tuple)): return qc[0] if len(qc) == 1 else None
    return qc

def labels_of(r):
    qc = r[QCLASS_COL]
    return list(qc) if isinstance(qc, (list, tuple)) else [qc]

def norm(a):
    return "" if a is None else re.sub(r"\s+", " ", str(a).strip().lower()).strip(" .")

def parse_original(s):
    if s is None: return []
    t = str(s).strip()
    for c in (t, t.replace("\\n", "\n"), t.replace("\\n", "")):
        try:
            v = json.loads(c)
            if isinstance(v, list): return [str(d.get("a", "")).strip() for d in v if isinstance(d, dict)]
        except Exception:
            pass
    return []

def tokf1(p, g):
    ps, gs = norm(p).split(), norm(g).split()
    if not ps and not gs: return 1.0
    if not ps or not gs: return 0.0
    sm = sum((Counter(ps) & Counter(gs)).values())
    if sm == 0: return 0.0
    P, R = sm / len(ps), sm / len(gs)
    return 2 * P * R / (P + R)

exem = defaultdict(lambda: defaultdict(set))
for r in rows:
    c = qclass(r)
    if c is None: continue
    a = parse_original(r["original"])
    if len(a) == 1:
        exem[c][norm(a[0])].add(norm(r[ANS_COL]))

def classify_asp(cat, text, tau=0.34):
    pn = set(norm(text).split()); t = norm(text); best, bk = None, (-1, -1, -1)
    for canon, pr in exem.get(cat, {}).items():
        para = max(tokf1(t, p) for p in pr)
        at = [x for x in canon.split(";") if x.strip()]
        cov = np.mean([1.0 if set(norm(x).split()) & pn else 0.0 for x in at]) if at else 0.0
        k = (round(para, 2), cov, -len(at))
        if k > bk: bk, best = k, canon
    return best if bk[0] >= tau else None

def aspects_and_golds(r):
    qc = labels_of(r); og = parse_original(r["original"])
    return list(zip(qc, [norm(a) for a in og])) if len(qc) == len(og) else None

L1_SCORES = {}
l1_count = sum(1 for r in rows if r.get("complexity") == 1 and aspects_and_golds(r))
l23_indices = [i for i, r in enumerate(rows) if r.get("complexity") in (2, 3) and aspects_and_golds(r)]

print(f"Total baris test: {len(rows)} | Baris L1 (deterministik): {l1_count} | Baris L2/L3 (LLM-Judge): {len(l23_indices)}\n")
print("=== SKOR DETERMINISTIK LEVEL-1 (5.496 PERTANYAAN TUNGGAL) ===")
for mname, preds in PREDS.items():
    m_l1 = {}
    vals = []
    for i, r in enumerate(rows):
        if r.get("complexity") != 1: continue
        ag = aspects_and_golds(r)
        if not ag: continue
        resp = preds.get(i, "")
        sc = {c: (1 if classify_asp(c, resp) == g else 0) for c, g in ag}
        m_l1[i] = sc
        vals.extend(sc.values())
    L1_SCORES[mname] = m_l1
    print(f"  {mname:26s} | L1 Deterministic Accuracy: {100.0 * np.mean(vals):6.2f}% (n={len(vals)})")

JUDGE_TMPL = (
    "You are a medical examiner. For EACH aspect, score 1 if the model's answer conveys that aspect's "
    "ground-truth MEANING, else 0. Judge by clinical meaning, NOT exact wording: paraphrases and "
    "negations count (e.g. 'No polyp identified' == 'none'; 'a single finding' == '1'). "
    "Return ONLY a JSON object mapping each aspect name to 0 or 1.\n\n"
    "### Example 1\n"
    "Question: What type of polyp is present and how many findings are visible?\n"
    "Model answer: No polypoid lesions identified, with a single abnormal finding noted.\n"
    "Aspects (name: ground_truth):\n- polyp_type: none\n- finding_count: 1\n"
    "JSON: {\"polyp_type\": 1, \"finding_count\": 1}\n\n"
    "### Example 2\n"
    "Question: Are there landmarks and is an instrument present?\n"
    "Model answer: No anatomical landmark is visible, and no instrument is present.\n"
    "Aspects (name: ground_truth):\n- landmark_presence: none\n- instrument_presence: none\n"
    "JSON: {\"landmark_presence\": 1, \"instrument_presence\": 1}\n\n"
    "### Example 3 (wrong)\n"
    "Question: What type of polyp?\n"
    "Model answer: A sessile polyp is present.\n"
    "Aspects (name: ground_truth):\n- polyp_type: paris ip\n"
    "JSON: {\"polyp_type\": 0}\n\n"
    "### Example 4\n"
    "Question: What colour is the abnormality and are box artifacts present?\n"
    "Model answer: The abnormality appears pink and red, and no box artifacts are seen.\n"
    "Aspects (name: ground_truth):\n- abnormality_color: pink; red\n- box_artifact_presence: no\n"
    "JSON: {\"abnormality_color\": 1, \"box_artifact_presence\": 1}\n\n"
    "### Example 5\n"
    "Question: How many instruments and what procedure?\n"
    "Model answer: Two instruments are visible during a colonoscopy.\n"
    "Aspects (name: ground_truth):\n- instrument_count: 2\n- procedure_type: colonoscopy\n"
    "JSON: {\"instrument_count\": 1, \"procedure_type\": 1}\n\n"
    "### Now grade this\n"
    "Question: <<Q>>\nModel answer: <<A>>\nAspects (name: ground_truth):\n<<ASP>>\nJSON:"
)

def build_prompt_text(idx, resp):
    r = rows[idx]
    ag = aspects_and_golds(r)
    asp = "\n".join(f"- {c}: {g}" for c, g in ag)
    return (JUDGE_TMPL.replace("<<Q>>", str(r["question"]))
                      .replace("<<A>>", str(resp))
                      .replace("<<ASP>>", asp))

def parse_judge_output(dec_text, idx, resp):
    ag = aspects_and_golds(rows[idx])
    try:
        s = dec_text[dec_text.find("{"):dec_text.rfind("}") + 1]
        d = json.loads(s)
        return {c: (1 if str(d.get(c, 0)).strip() in ("1", "1.0", "true", "True") else 0) for c, _ in ag}
    except Exception:
        return {c: (1 if classify_asp(c, resp) == g else 0) for c, g in ag}

def run_deduplicated_judge(judge_tag, jmodel, jtok, models_dict, bs=32, save_every=400):
    jtok.padding_side = "left"
    if jtok.pad_token is None:
        jtok.pad_token = jtok.eos_token

    pool_path = DUAL_JDIR / f"global_pool_{judge_tag}.json"
    p2_pool_path = P2_JDIR / f"global_pool_{judge_tag}.json"
    sh_pool_path = SHARED_JDIR / f"global_pool_{judge_tag}.json"
    pool = {}
    for pp in [sh_pool_path, p2_pool_path, pool_path]:
        if pp.exists():
            try:
                pool.update(json.loads(pp.read_text(encoding="utf-8")))
            except Exception:
                pass
    print(f"[{judge_tag}] Memuat Global Deduplication Pool: {len(pool)} entri unik tersedia.")

    for mname, mpreds in models_dict.items():
        for mc_dir in [SHARED_JDIR, P2_JDIR, DUAL_JDIR]:
            m_cache = mc_dir / f"judge_{judge_tag}_{mname}.json"
            if m_cache.exists():
                try:
                    jd_m = json.loads(m_cache.read_text(encoding="utf-8"))
                    for k_str, val in jd_m.items():
                        if val and all(v is not None for v in val.values()):
                            idx = int(k_str)
                            ans_key = f"{idx}||{norm(mpreds.get(idx, ''))}"
                            if ans_key not in pool:
                                pool[ans_key] = val
                except Exception:
                    pass

    todo_map = {}
    total_needed = len(models_dict) * len(l23_indices)
    for mname, mpreds in models_dict.items():
        for idx in l23_indices:
            raw_resp = mpreds.get(idx, "")
            ans_key = f"{idx}||{norm(raw_resp)}"
            if ans_key not in pool and ans_key not in todo_map:
                todo_map[ans_key] = (idx, raw_resp)

    todo_list = list(todo_map.items())
    print(f"[{judge_tag}] Total kebutuhan evaluasi: {total_needed} ({len(models_dict)} model x {len(l23_indices)} baris L2/L3)")
    print(f"[{judge_tag}] Sudah ada di Cache/Pool: {total_needed - len(todo_list)} | Sisa unik yang harus di-run di GPU: {len(todo_list)}")

    if len(todo_list) > 0:
        todo_list.sort(key=lambda x: len(str(rows[x[1][0]]["question"])) + len(str(x[1][1])))
        pbar = tqdm(range(0, len(todo_list), bs), desc=f"Judging [{judge_tag}] (bs={bs})")
        processed_since_save = 0
        for start_i in pbar:
            chunk = todo_list[start_i : start_i + bs]
            prompts = []
            for ans_key, (idx, raw_resp) in chunk:
                p_txt = build_prompt_text(idx, raw_resp)
                chat_txt = jtok.apply_chat_template([{"role": "user", "content": p_txt}], tokenize=False, add_generation_prompt=True)
                prompts.append(chat_txt)

            inp = jtok(prompts, return_tensors="pt", padding=True, truncation=True, max_length=1024).to(jmodel.device)
            with torch.no_grad():
                out = jmodel.generate(**inp, max_new_tokens=64, do_sample=False, pad_token_id=jtok.pad_token_id)

            plen = inp["input_ids"].shape[1]
            for (ans_key, (idx, raw_resp)), o in zip(chunk, out):
                dec = jtok.decode(o[plen:], skip_special_tokens=True)
                pool[ans_key] = parse_judge_output(dec, idx, raw_resp)

            processed_since_save += len(chunk)
            if processed_since_save >= save_every or (start_i + bs >= len(todo_list)):
                pool_path.write_text(json.dumps(pool))
                if P2_JDIR.exists():
                    p2_pool_path.write_text(json.dumps(pool))
                done_now = min(start_i + bs, len(todo_list))
                print(f"  [Checkpoint {judge_tag}] {done_now}/{len(todo_list)} ({100*done_now/len(todo_list):.1f}%) tersimpan | Total Pool: {len(pool)}", flush=True)
                processed_since_save = 0

    all_model_judgments = {}
    for mname, mpreds in models_dict.items():
        m_jud = {}
        for idx in l23_indices:
            raw_resp = mpreds.get(idx, "")
            ans_key = f"{idx}||{norm(raw_resp)}"
            m_jud[idx] = pool[ans_key]
        m_cache = DUAL_JDIR / f"judge_{judge_tag}_{mname}.json"
        m_cache.write_text(json.dumps({str(k): v for k, v in m_jud.items()}))
        all_model_judgments[mname] = m_jud
        print(f"  -> Tersimpan lengkap: {m_cache.name} ({len(m_jud)}/10459 baris L2/L3)")

    return all_model_judgments

Total baris test: 15955 | Baris L1 (deterministik): 5496 | Baris L2/L3 (LLM-Judge): 10459

=== SKOR DETERMINISTIK LEVEL-1 (5.496 PERTANYAAN TUNGGAL) ===
  A0_VQAonly_r32             | L1 Deterministic Accuracy:  79.99% (n=5496)
  E_CLBRIDGE_r32             | L1 Deterministic Accuracy:  81.20% (n=5496)
  Gautam_Qwen2.5VL_7B        | L1 Deterministic Accuracy:  84.43% (n=5496)
  M1_GeoPE_r32               | L1 Deterministic Accuracy:  79.95% (n=5496)
  M2_GeoDiff_r32             | L1 Deterministic Accuracy:  81.62% (n=5496)
  M3_StaticGate_r32          | L1 Deterministic Accuracy:  82.62% (n=5496)
  M4_GeoFiLM_r32             | L1 Deterministic Accuracy:  83.79% (n=5496)
  Bridge_B1_M4_CanonBridge   | L1 Deterministic Accuracy:  84.24% (n=5496)
  Ortho_O2_Ortho_Unified_Full | L1 Deterministic Accuracy:  83.73% (n=5496)


## [J2] Eksekusi JUDGE 1: `Qwen/Qwen2.5-7B-Instruct`

In [7]:
# [J2] Load Qwen/Qwen2.5-7B-Instruct & Jalankan Evaluasi Penuh pada Sisa Baris Unik Ortho-Unified
import torch, gc
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

QWEN_JID = "Qwen/Qwen2.5-7B-Instruct"
vram_gb = torch.cuda.get_device_properties(0).total_memory / 1e9 if torch.cuda.is_available() else 0
USE_4BIT_QWEN = (vram_gb < 20.0)

print(f"Memuat Judge 1: {QWEN_JID} | VRAM={vram_gb:.1f}GB | 4-bit={USE_4BIT_QWEN}")
jtok_qwen = AutoTokenizer.from_pretrained(QWEN_JID)
if USE_4BIT_QWEN:
    bnb_cfg = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_compute_dtype=torch.bfloat16, bnb_4bit_quant_type="nf4")
    jmodel_qwen = AutoModelForCausalLM.from_pretrained(QWEN_JID, quantization_config=bnb_cfg, device_map="auto").eval()
else:
    jmodel_qwen = AutoModelForCausalLM.from_pretrained(QWEN_JID, torch_dtype=torch.bfloat16, device_map="auto").eval()

QWEN_JUDGMENTS = run_deduplicated_judge(
    judge_tag="qwen25_7b",
    jmodel=jmodel_qwen,
    jtok=jtok_qwen,
    models_dict=PREDS,
    bs=32 if vram_gb >= 15 else 16,
    save_every=400
)

del jmodel_qwen, jtok_qwen
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
print("Judge 1 (Qwen2.5-7B) selesai 100% dan memori GPU telah dibersihkan!")

Memuat Judge 1: Qwen/Qwen2.5-7B-Instruct | VRAM=42.4GB | 4-bit=False


Sliding Window Attention is enabled but not implemented for `sdpa`; unexpected results may be encountered.


Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

[qwen25_7b] Memuat Global Deduplication Pool: 67360 entri unik tersedia.
[qwen25_7b] Total kebutuhan evaluasi: 94131 (9 model x 10459 baris L2/L3)
[qwen25_7b] Sudah ada di Cache/Pool: 89966 | Sisa unik yang harus di-run di GPU: 4165


Judging [qwen25_7b] (bs=32):   0%|          | 0/131 [00:00<?, ?it/s]

  [Checkpoint qwen25_7b] 416/4165 (10.0%) tersimpan | Total Pool: 67776
  [Checkpoint qwen25_7b] 832/4165 (20.0%) tersimpan | Total Pool: 68192
  [Checkpoint qwen25_7b] 1248/4165 (30.0%) tersimpan | Total Pool: 68608
  [Checkpoint qwen25_7b] 1664/4165 (40.0%) tersimpan | Total Pool: 69024
  [Checkpoint qwen25_7b] 2080/4165 (49.9%) tersimpan | Total Pool: 69440
  [Checkpoint qwen25_7b] 2496/4165 (59.9%) tersimpan | Total Pool: 69856
  [Checkpoint qwen25_7b] 2912/4165 (69.9%) tersimpan | Total Pool: 70272
  [Checkpoint qwen25_7b] 3328/4165 (79.9%) tersimpan | Total Pool: 70688
  [Checkpoint qwen25_7b] 3744/4165 (89.9%) tersimpan | Total Pool: 71104
  [Checkpoint qwen25_7b] 4160/4165 (99.9%) tersimpan | Total Pool: 71520
  [Checkpoint qwen25_7b] 4165/4165 (100.0%) tersimpan | Total Pool: 71525
  -> Tersimpan lengkap: judge_qwen25_7b_A0_VQAonly_r32.json (10459/10459 baris L2/L3)
  -> Tersimpan lengkap: judge_qwen25_7b_E_CLBRIDGE_r32.json (10459/10459 baris L2/L3)
  -> Tersimpan lengkap: ju

## [J3] Eksekusi JUDGE 2: `NousResearch/Meta-Llama-3.1-8B-Instruct`

In [8]:
# [J3] Load NousResearch/Meta-Llama-3.1-8B-Instruct & Jalankan Evaluasi Penuh pada Sisa Baris Unik Ortho-Unified
import torch, gc
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

LLAMA_JID = "NousResearch/Meta-Llama-3.1-8B-Instruct"
vram_gb = torch.cuda.get_device_properties(0).total_memory / 1e9 if torch.cuda.is_available() else 0
USE_4BIT_LLAMA = (vram_gb < 22.0)

print(f"Memuat Judge 2: {LLAMA_JID} | VRAM={vram_gb:.1f}GB | 4-bit={USE_4BIT_LLAMA}")
jtok_llama = AutoTokenizer.from_pretrained(LLAMA_JID)
if USE_4BIT_LLAMA:
    bnb_cfg = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_compute_dtype=torch.bfloat16, bnb_4bit_quant_type="nf4")
    jmodel_llama = AutoModelForCausalLM.from_pretrained(LLAMA_JID, quantization_config=bnb_cfg, device_map="auto").eval()
else:
    jmodel_llama = AutoModelForCausalLM.from_pretrained(LLAMA_JID, torch_dtype=torch.bfloat16, device_map="auto").eval()

LLAMA_JUDGMENTS = run_deduplicated_judge(
    judge_tag="llama31_8b",
    jmodel=jmodel_llama,
    jtok=jtok_llama,
    models_dict=PREDS,
    bs=32 if vram_gb >= 15 else 16,
    save_every=400
)

del jmodel_llama, jtok_llama
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
print("Judge 2 (Llama-3.1-8B) selesai 100% dan memori GPU telah dibersihkan!")

Memuat Judge 2: NousResearch/Meta-Llama-3.1-8B-Instruct | VRAM=42.4GB | 4-bit=False


Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

[llama31_8b] Memuat Global Deduplication Pool: 67360 entri unik tersedia.
[llama31_8b] Total kebutuhan evaluasi: 94131 (9 model x 10459 baris L2/L3)
[llama31_8b] Sudah ada di Cache/Pool: 89966 | Sisa unik yang harus di-run di GPU: 4165


Judging [llama31_8b] (bs=32):   0%|          | 0/131 [00:00<?, ?it/s]

  [Checkpoint llama31_8b] 416/4165 (10.0%) tersimpan | Total Pool: 67776
  [Checkpoint llama31_8b] 832/4165 (20.0%) tersimpan | Total Pool: 68192
  [Checkpoint llama31_8b] 1248/4165 (30.0%) tersimpan | Total Pool: 68608
  [Checkpoint llama31_8b] 1664/4165 (40.0%) tersimpan | Total Pool: 69024
  [Checkpoint llama31_8b] 2080/4165 (49.9%) tersimpan | Total Pool: 69440
  [Checkpoint llama31_8b] 2496/4165 (59.9%) tersimpan | Total Pool: 69856
  [Checkpoint llama31_8b] 2912/4165 (69.9%) tersimpan | Total Pool: 70272
  [Checkpoint llama31_8b] 3328/4165 (79.9%) tersimpan | Total Pool: 70688
  [Checkpoint llama31_8b] 3744/4165 (89.9%) tersimpan | Total Pool: 71104
  [Checkpoint llama31_8b] 4160/4165 (99.9%) tersimpan | Total Pool: 71520
  [Checkpoint llama31_8b] 4165/4165 (100.0%) tersimpan | Total Pool: 71525
  -> Tersimpan lengkap: judge_llama31_8b_A0_VQAonly_r32.json (10459/10459 baris L2/L3)
  -> Tersimpan lengkap: judge_llama31_8b_E_CLBRIDGE_r32.json (10459/10459 baris L2/L3)
  -> Tersimpa

## [J4] Kompilasi Komprehensif: 18 Kategori $\times$ 3 Level Kompleksitas (`31.622` Aspek), Cohen's $\kappa$, & Uji McNemar

In [9]:
# [J4] Kompilasi metrik Hybrid (L1 Deterministik + L2/L3 LLM-Judge), Cohen's Kappa, & Uji McNemar Berpasangan
def compile_hybrid_scores(judge_tag):
    model_per_cat = {}
    model_per_cat_lv = {}
    model_per_lv = {}
    model_row_scores = {}

    for mname in PREDS:
        m_cache = DUAL_JDIR / f"judge_{judge_tag}_{mname}.json"
        assert m_cache.exists(), f"Cache judge belum ada: {m_cache}"
        jd = {int(k): v for k, v in json.loads(m_cache.read_text(encoding="utf-8")).items()}
        l1_d = L1_SCORES[mname]

        per_c = defaultdict(list)
        per_clv = defaultdict(list)
        per_lv = defaultdict(list)
        row_sc = {}

        for i, r in enumerate(rows):
            ag = aspects_and_golds(r)
            if not ag: continue
            lv = r.get("complexity")
            res = l1_d.get(i) if lv == 1 else jd.get(i)
            if not res: continue
            vals = []
            for c, _ in ag:
                v = res.get(c)
                if v is not None:
                    per_c[c].append(int(v))
                    per_clv[(c, lv)].append(int(v))
                    per_lv[lv].append(int(v))
                    vals.append(int(v))
            if vals:
                row_sc[i] = float(np.mean(vals))

        model_per_cat[mname] = per_c
        model_per_cat_lv[mname] = per_clv
        model_per_lv[mname] = per_lv
        model_row_scores[mname] = row_sc

    summary_rows = []
    for mname in PREDS:
        per_lv = model_per_lv[mname]
        all_v = per_lv[1] + per_lv[2] + per_lv[3]
        l23_v = per_lv[2] + per_lv[3]
        summary_rows.append({
            "Judge": judge_tag,
            "Model": mname,
            "Overall (%)": round(100 * np.mean(all_v), 2),
            "L1_Det (%)": round(100 * np.mean(per_lv[1]), 2),
            "L2_Judge (%)": round(100 * np.mean(per_lv[2]), 2),
            "L3_Judge (%)": round(100 * np.mean(per_lv[3]), 2),
            "L2+L3_Judge (%)": round(100 * np.mean(l23_v), 2),
            "Total_Aspects": len(all_v)
        })
    df_sum = pd.DataFrame(summary_rows)

    ref_key = next(iter(PREDS.keys()))
    all_cats = sorted(model_per_cat[ref_key].keys())
    cat_rows = []
    for c in all_cats:
        row_d = {"Category": c, "N_Aspects": len(model_per_cat[ref_key][c])}
        for mname in PREDS:
            row_d[f"{mname}_ALL"] = round(100 * np.mean(model_per_cat[mname][c]), 2) if model_per_cat[mname][c] else np.nan
            for lv in (1, 2, 3):
                lst = model_per_cat_lv[mname].get((c, lv), [])
                row_d[f"{mname}_L{lv}"] = round(100 * np.mean(lst), 2) if lst else np.nan
        cat_rows.append(row_d)
    df_cat = pd.DataFrame(cat_rows)

    df_sum.to_csv(DUAL_JDIR / f"summary_by_level_{judge_tag}.csv", index=False)
    df_cat.to_csv(DUAL_JDIR / f"comprehensive_18cats_3levels_{judge_tag}.csv", index=False)
    return df_sum, df_cat, model_row_scores

df_sum_qwen, df_cat_qwen, row_sc_qwen   = compile_hybrid_scores("qwen25_7b")
df_sum_llama, df_cat_llama, row_sc_llama = compile_hybrid_scores("llama31_8b")

print("=== [JUDGE 1: QWEN2.5-7B] RINGKASAN SKOR HYBRID PER LEVEL ===")
print(df_sum_qwen.to_string(index=False))

print("\n=== [JUDGE 2: LLAMA-3.1-8B] RINGKASAN SKOR HYBRID PER LEVEL ===")
print(df_sum_llama.to_string(index=False))

print("\n=== [PERBANDINGAN 18 KATEGORI (ALL)] QWEN-JUDGE vs LLAMA-JUDGE ===")
cols_all = ["Category", "N_Aspects"] + [f"{m}_ALL" for m in PREDS]
print("--- Qwen2.5-7B Judge ---")
print(df_cat_qwen[cols_all].to_string(index=False))
print("\n--- Llama-3.1-8B Judge ---")
print(df_cat_llama[cols_all].to_string(index=False))

# Uji Signifikansi McNemar pada 31.622 Aspek untuk PRIMARY_MODEL vs Seluruh Model Lain
def mcnemar_all_aspects(judge_tag, target_m):
    jt = {int(k): v for k, v in json.loads((DUAL_JDIR / f"judge_{judge_tag}_{target_m}.json").read_text(encoding="utf-8")).items()}
    l1_t = L1_SCORES[target_m]
    mcn_list = []
    for ref_m in PREDS:
        if ref_m == target_m: continue
        jr = {int(k): v for k, v in json.loads((DUAL_JDIR / f"judge_{judge_tag}_{ref_m}.json").read_text(encoding="utf-8")).items()}
        l1_r = L1_SCORES[ref_m]
        b, c, n_tot = 0, 0, 0
        for i, r in enumerate(rows):
            ag = aspects_and_golds(r)
            if not ag: continue
            lv = r.get("complexity")
            rt = l1_t.get(i) if lv == 1 else jt.get(i)
            rr = l1_r.get(i) if lv == 1 else jr.get(i)
            if not rt or not rr: continue
            for asp_c, _ in ag:
                vt, vr = rt.get(asp_c), rr.get(asp_c)
                if vt is not None and vr is not None:
                    n_tot += 1
                    if vr == 1 and vt == 0: b += 1
                    elif vr == 0 and vt == 1: c += 1
        pval = float(binomtest(min(b, c), b + c, 0.5).pvalue) if (b + c) > 0 else 1.0
        mcn_list.append({
            "Judge": judge_tag,
            "Comparison": f"{target_m} vs {ref_m}",
            "N_Aspects": n_tot,
            "Fix (c)": c,
            "Break (b)": b,
            "Net (c-b)": c - b,
            "Net_Gain (%)": round(100.0 * (c - b) / max(n_tot, 1), 2),
            "McNemar_p": pval
        })
    return pd.DataFrame(mcn_list)

df_mcn_primary = pd.concat([
    mcnemar_all_aspects("qwen25_7b", PRIMARY_MODEL),
    mcnemar_all_aspects("llama31_8b", PRIMARY_MODEL)
], ignore_index=True)
print(f"\n=== UJI MCNEMAR HYBRID 31.622 ASPEK ({PRIMARY_MODEL} vs MODEL LAIN) ===")
print(df_mcn_primary.to_string(index=False))
df_mcn_primary.to_csv(DUAL_JDIR / f"mcnemar_dual_judge_{PRIMARY_MODEL}.csv", index=False)

=== [JUDGE 1: QWEN2.5-7B] RINGKASAN SKOR HYBRID PER LEVEL ===
    Judge                       Model  Overall (%)  L1_Det (%)  L2_Judge (%)  L3_Judge (%)  L2+L3_Judge (%)  Total_Aspects
qwen25_7b              A0_VQAonly_r32        82.44       79.99         82.35         83.36            82.95          31622
qwen25_7b              E_CLBRIDGE_r32        84.76       81.20         84.56         86.14            85.50          31622
qwen25_7b         Gautam_Qwen2.5VL_7B        85.50       84.43         83.51         87.21            85.72          31622
qwen25_7b                M1_GeoPE_r32        84.13       79.95         83.98         85.70            85.01          31622
qwen25_7b              M2_GeoDiff_r32        84.65       81.62         84.27         85.97            85.29          31622
qwen25_7b           M3_StaticGate_r32        84.74       82.62         84.03         85.97            85.19          31622
qwen25_7b              M4_GeoFiLM_r32        85.29       83.79         84.22 